<a href="https://colab.research.google.com/github/NikhilGeorge01/DeepLearningPractice/blob/main/regularization(week5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split



In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

X_train_flat = X_train.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)



In [ ]:
def build_mlp(l2_reg=0.0, dropout_rate=0.0):
    model = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_reg)),
        layers.Dropout(dropout_rate),
        layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_reg)),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model



In [ ]:
results = {}

# Base model
model = build_mlp()
model.fit(X_train_flat, y_train, epochs=10, batch_size=128, verbose=0)
results["Base"] = model.evaluate(X_test_flat, y_test)[1]



313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9743 - loss: 0.1094


In [ ]:
# L2 Regularization
model = build_mlp(l2_reg=0.001)
model.fit(X_train_flat, y_train, epochs=10, batch_size=128, verbose=0)
results["L2"] = model.evaluate(X_test_flat, y_test, verbose=0)[1]



In [ ]:
# Dropout
model = build_mlp(dropout_rate=0.3)
model.fit(X_train_flat, y_train, epochs=10, batch_size=128, verbose=0)
results["Dropout"] = model.evaluate(X_test_flat, y_test, verbose=0)[1]



In [ ]:
# Early Stopping
model = build_mlp()
callback = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)
model.fit(X_train_flat, y_train, epochs=30, batch_size=128,
          validation_split=0.1, callbacks=[callback], verbose=0)
results["EarlyStopping"] = model.evaluate(X_test_flat, y_test, verbose=0)[1]



In [ ]:
# Adding Noise
X_train_noise = X_train_flat + 0.1 * np.random.normal(size=X_train_flat.shape)
model = build_mlp()
model.fit(X_train_noise, y_train, epochs=10, batch_size=128, verbose=0)
results["InputNoise"] = model.evaluate(X_test_flat, y_test, verbose=0)[1]



In [ ]:
# Dataset Augmentation (image-based then flattened)
datagen = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# Wrapper generator to flatten the images
def flatten_generator(generator):
    for X_batch, y_batch in generator:
        yield X_batch.reshape(X_batch.shape[0], -1), y_batch

model = build_mlp()
batch_size = 128
aug_generator = datagen.flow(X_train, y_train, batch_size=batch_size)

# Calculate steps per epoch
steps_per_epoch = len(X_train) // batch_size

model.fit(flatten_generator(aug_generator), epochs=10, steps_per_epoch=steps_per_epoch, verbose=0)
results["Augmentation"] = model.evaluate(X_test_flat, y_test, verbose=0)[1]


In [ ]:
# Ensemble
models = []
for _ in range(3):
    m = build_mlp()
    m.fit(X_train_flat, y_train, epochs=10, batch_size=128, verbose=0)
    models.append(m)

preds = np.mean([m.predict(X_test_flat, verbose=0) for m in models], axis=0)
ensemble_acc = np.mean(np.argmax(preds, axis=1) == y_test)
results["Ensemble"] = ensemble_acc

for k,v in results.items():
    print(k, "Accuracy:", round(v,4))

Base Accuracy: 0.9781
L2 Accuracy: 0.976
Dropout Accuracy: 0.9793
EarlyStopping Accuracy: 0.9734
InputNoise Accuracy: 0.9785
Augmentation Accuracy: 0.9831
Ensemble Accuracy: 0.9817
